In [30]:
import pandas as pd
# 直接从UCI machine learning dataset中读取电力预测数据
#df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv")
df = pd.read_csv('energydata_complete.csv')

In [31]:
from gluonts.dataset.common import ListDataset
# 选取部分特征
features = ["date", "Appliances", "T_out", "Press_mm_hg", "RH_out", "Windspeed", "Tdewpoint", "Visibility"]
df_input = df[features].set_index('date')
# 分割训练数据和测试数据
train_time_end = "2016-05-10 00:00:00"
prediction_length = 144 # 设置预测的长度
training_data = ListDataset(
    [{"start": df_input.index[0], "target": df_input.Appliances[:train_time_end]}],
    freq = "10min"
) # 注意这里metadata freq是必须的
# 准备两个不同时间点的测试数据集
test_data = ListDataset(
    [
        {"start": df_input.index[0], "target": df_input.Appliances[:"2016-05-11 00:00:00"]},
        {"start": df_input.index[0], "target": df_input.Appliances[:"2016-05-15 00:00:00"]}
    ],freq="10min"
)

In [33]:
import numpy as np
from gluonts.model.deepar import DeepAREstimator
from gluonts.mx.trainer import Trainer

# 建立模型
estimator = DeepAREstimator(freq="10min",
                            context_length=720,  # 这是RNN的窗口大小
                            prediction_length=prediction_length,
                            num_layers=2,
                            num_cells=128,
                            cell_type="lstm",
                            num_batches_per_epoch=50,
                            trainer=Trainer(epochs=8))


/Users/wangfeng/runtime/pyenv/global/lib/python3.8/site-packages/mxnet/numpy/utils.py:37: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.  (This may have returned Python scalars in past versions.
  bool = onp.bool


AttributeError: module 'numpy' has no attribute 'bool'

In [ ]:
#对模型进行训练
predictor = estimator.train(training_data = training_data)

In [ ]:
# 预测和模型评价
from gluonts.evaluation.backtest import make_evaluation_predictions
# 在测试集上进行预测
forcast_it, ts_it = make_evaluation_predictions(
    dataset = test_data,
    predictor = predictor,
    num_samples = 100,
)
forecasts = list(forcast_it)
tss = list(ts_it)

In [ ]:
from matplotlib import pyplot as plt


# 我们还需要一个函数对预测进行可视化，包括画出真值、预测值和不同置信度的置信区间
def plot_prob_forecasts(ts_entry, forecast_entry):
    plot_length = 200
    prob_intervals = (50.0, 95.0)
    legend = ["True values", "median prediction"] + [f"{k}% prob interval" for k in prob_intervals][::-1]

    fig = plt.figure(figsize = (10, 5), dpi = 120)
    ax = plt.gca()
    ts_entry[-plot_length:].plot(ax = ax)
    forecast_entry.plot(prediction_intervals = prob_intervals, color = 'g')
    plt.legend(legend)
    plt.show()